# XManager with Vertex Training Cluster

This notebook demonstrates how to use XManager to launch and manage **distributed training jobs** on Slurm-based Vertex Training Clusters.

## What You'll Learn

1. **Setup** - Initialize XManager in a notebook environment
2. **Configuration** - Set cluster connection and job parameters
3. **Cluster Configs** - Pre-configured NCCL settings for different GPU types
4. **PyTorch/NeMo Training** - Launch torchrun-based distributed training
5. **Experiment Tracking** - List, query, and monitor experiments
6. **Job Management** - Check status, view logs, cancel jobs

## Supported Cluster Types

| Cluster Type | GPU | GPUs/Node | Network |
|-------------|-----|-----------|----------|
| `hcc-a3m` | H100 | 8 | TCPXO |
| `hcc-a3u` | H200 | 8 | gIB |
| `hcc-a4` | B200 | 8 | gIB |
| `hcc-a3h` | H100 | 8 | gIB |

---
## 1. Setup and Installation

Install dependencies and initialize XManager for notebook use.

In [ ]:
# Install dependencies
!pip install nest_asyncio pandas -q

# For development, install from local source
# !pip install -e .

In [1]:
# Core imports
from xmanager import xm
from xmanager import xm_local

# For displaying results
import pandas as pd
from IPython.display import display, HTML

# Initialize absl flags (required for XManager in notebooks)
import sys
from absl import flags

# Parse flags with empty argv to initialize XManager
# This is needed because XManager uses absl flags internally
if not flags.FLAGS.is_parsed():
    flags.FLAGS(sys.argv[:1])  # Only pass the script name, ignore notebook args

# Enable nested asyncio for Jupyter compatibility
# XManager uses asyncio internally, which conflicts with Jupyter's event loop

import nest_asyncio
nest_asyncio.apply()

print("XManager initialized successfully!")

XManager initialized successfully!


---
## 2. Configuration

Set your cluster connection details and job parameters.

### Key Concepts:
- **work_dir**: Directory on cluster containing your scripts, data, and where logs are written. This is mounted at the same path inside the container.
- **container_image**: Path to a squashfs container image (.sqsh) on the cluster
- **training_script**: Your training script (relative to work_dir)

In [2]:
# =============================================================================
# CLUSTER CONNECTION - Update these for your environment
# =============================================================================

# SSH connection to cluster login node
LOGIN_NODE = "vmdsa3u04-login-001"  # Cluster login node name
USE_GCLOUD_SSH = True  # Use gcloud compute ssh (vs direct ssh)
SSH_HOSTNAME = "nic0.vmdsa3u04-login-001.europe-west4-a.c.ai-infra-recipe-validation.internal.gcpnode.com"

# =============================================================================
# CLUSTER SETTINGS
# =============================================================================

CLUSTER_TYPE = "hcc-a3u"  # Options: hcc-a3m, hcc-a3u, hcc-a4, hcc-a3h
PARTITION = "a3u"         # Slurm partition
ACCOUNT = "aaie"          # Slurm account (optional)
NUM_NODES = 4             # Number of nodes (8 for Llama-3.1-70B)
TIME_LIMIT = None         # Time limit (e.g., "1:00:00"). None = no limit directive

# =============================================================================
# PATHS ON CLUSTER
# =============================================================================

# Lustre instance name (update for your environment)
LUSTRE_INSTANCE_NAME = "ls1-europe-west4-a"

# Container image (squashfs format) - standard location for shared images
CONTAINER_IMAGE = "/home/common/images/nemo.25.07.sqsh"

# Logs path on Managed Lustre - checkpoints are saved here
LOGS_PATH = f"/mnt/lustre/{LUSTRE_INSTANCE_NAME}/jobs"

# Work directory - contains training scripts
# This is mounted at the same path inside the container
WORK_DIR = "/home/abhishekbhgwt_google_com/vertexai-mds/nemo"

# Training script (relative to WORK_DIR)
TRAINING_SCRIPT = "pretrain.py"

# =============================================================================
# TENSORBOARD SETTINGS (for Vertex AI TensorBoard integration)
# =============================================================================

# GCP project for TensorBoard (important: must match where your TB instance exists)
TENSORBOARD_PROJECT = "ai-infra-recipe-validation"

# Vertex AI TensorBoard instance display name (will be looked up or created)
TENSORBOARD_NAME = "ai-infra-europe-west4-tb"

# GCS bucket path for TensorBoard logs (bucket-name format, no gs:// or /gcs/)
# VMDS uploads logs from here to Vertex AI TensorBoard
TENSORBOARD_GCS_PATH = "ai-infra-gcs-europe-west4"

# Vertex AI region for TensorBoard
TENSORBOARD_REGION = "europe-west4"

print("Configuration loaded!")
print(f"  Cluster: {CLUSTER_TYPE}")
print(f"  Partition: {PARTITION}")
print(f"  Nodes: {NUM_NODES}")
print(f"  Container: {CONTAINER_IMAGE}")
print(f"  Work dir: {WORK_DIR}")
print(f"  Checkpoints: {LOGS_PATH}")
print(f"  TensorBoard Project: {TENSORBOARD_PROJECT}")
print(f"  TensorBoard: {TENSORBOARD_NAME}")
print(f"  TensorBoard GCS: {TENSORBOARD_GCS_PATH}")
print(f"  Training script: {WORK_DIR}/{TRAINING_SCRIPT}")

Configuration loaded!
  Cluster: hcc-a3u
  Partition: a3u
  Nodes: 4
  Container: /home/common/images/nemo.25.07.sqsh
  Work dir: /home/abhishekbhgwt_google_com/vertexai-mds/nemo
  Checkpoints: /mnt/lustre/ls1-europe-west4-a/jobs
  TensorBoard Project: ai-infra-recipe-validation
  TensorBoard: ai-infra-europe-west4-tb
  TensorBoard GCS: ai-infra-gcs-europe-west4
  Training script: /home/abhishekbhgwt_google_com/vertexai-mds/nemo/pretrain.py


---
## 3. Cluster Configuration Details

Each cluster type has **pre-configured NCCL settings** optimized for its networking hardware.

XManager automatically applies these settings when you specify `cluster_type`.

In [3]:
# Create a minimal executor to inspect cluster config
executor = xm_local.VertexTrainingCluster(
    cluster_type=CLUSTER_TYPE,
    partition=PARTITION,
)

# Get the cluster configuration
cluster_config = executor.get_cluster_config()

print(f"Cluster Configuration for {CLUSTER_TYPE}")
print("=" * 50)
print(f"GPU Type: {cluster_config.gpu_type}")
print(f"GPUs per Node: {cluster_config.gpus_per_node}")
print(f"Total GPUs (with {NUM_NODES} nodes): {NUM_NODES * cluster_config.gpus_per_node}")
print()
print("NCCL Environment Variables (auto-configured):")
for key, value in cluster_config.nccl_env_vars.items():
    print(f"  {key}={value}")

Cluster Configuration for hcc-a3u
GPU Type: H200
GPUs per Node: 8
Total GPUs (with 4 nodes): 32

NCCL Environment Variables (auto-configured):


---
## 4. PyTorch/NeMo Distributed Training

Launch a distributed PyTorch training job using `torchrun`.

### How it works:

1. **Executor Setup**: Configure cluster, container, mounts, environment
2. **Job Creation**: Specify `torchrun` with distributed training args
3. **Sbatch Generation**: XManager generates an sbatch script with:
   - Head node discovery via `scontrol show hostnames`
   - NCCL configuration for the cluster type
   - Container execution via `srun --container-image`
4. **Submission**: Script is submitted via SSH to the login node

### Key Pattern: `xm.ShellSafeArg`

Use `xm.ShellSafeArg` for arguments containing shell variables (like `${SLURM_NNODES}`) that should expand at runtime, not be quoted.

In [4]:
# =============================================================================
# STEP 1: Configure the Executor
# =============================================================================

# Container mounts - mount work_dir and logs_path at same paths inside container
# Note: gIB mount (/usr/local/gib) is automatically added by XManager
# based on cluster_config.nccl_dir, so we don't add it here
container_mounts = [
    f"{WORK_DIR}:{WORK_DIR}",
    f"{LOGS_PATH}:{LOGS_PATH}",
]

# Environment variables for training
env_vars = {
    'NEMORUN_HOME': WORK_DIR,
    'GIB_PATH': '/usr/local/gib',
    'NCCL_SOCKET_IFNAME': 'enp0s19,enp192s20',
    'NCCL_DEBUG': 'VERSION',
    'CUDA_DEVICE_MAX_CONNECTIONS': '1',
    'OMP_NUM_THREADS': '12',
}

# Environment variables to pass through to container via --container-env
# These must be listed here to be available inside the container
container_env_passthrough = ['NCCL_SOCKET_IFNAME', 'NCCL_DEBUG', 'OMP_NUM_THREADS']

# TensorBoard integration (optional) - enables native VTC TensorBoard support
# VMDS will set AIP_TENSORBOARD_LOG_DIR inside the job
tensorboard = xm_local.TensorboardCapability(
    name=TENSORBOARD_NAME,  # Will be looked up/created automatically
    base_output_directory=TENSORBOARD_GCS_PATH,  # GCS bucket path
)

# Create the executor
executor = xm_local.VertexTrainingCluster(
    # Cluster settings
    cluster_type=CLUSTER_TYPE,
    partition=PARTITION,
    account=ACCOUNT,
    time_limit=TIME_LIMIT,
    
    # Resource requirements (replicas = number of nodes)
    requirements=xm.JobRequirements(replicas=NUM_NODES),
    
    # SSH connection to login node
    login_node=LOGIN_NODE,
    use_gcloud_ssh=USE_GCLOUD_SSH,
    ssh_hostname=SSH_HOSTNAME,
    
    # Working directory (mounted at same path in container)
    work_dir=WORK_DIR,
    
    # Container configuration
    container_image=CONTAINER_IMAGE,
    container_mounts=container_mounts,
    container_env_passthrough=container_env_passthrough,
    
    # Use MPI for PyTorch distributed training
    use_mpi=True,
    
    # Environment variables
    env_vars=env_vars,
    
    # Stream output to console
    stream_output=True,
    
    # TensorBoard integration
    tensorboard=tensorboard,
    tensorboard_region=TENSORBOARD_REGION,
    tensorboard_project=TENSORBOARD_PROJECT,
)

print("Executor configured!")
print(f"  Container: {CONTAINER_IMAGE}")
print(f"  Mounts: {container_mounts}")
print(f"  TensorBoard: {TENSORBOARD_NAME}")
print(f"  TensorBoard Project: {TENSORBOARD_PROJECT}")
print(f"  TensorBoard Region: {TENSORBOARD_REGION}")
print(f"  TensorBoard GCS: {TENSORBOARD_GCS_PATH}")

Executor configured!
  Container: /home/common/images/nemo.25.07.sqsh
  Mounts: ['/home/abhishekbhgwt_google_com/vertexai-mds/nemo:/home/abhishekbhgwt_google_com/vertexai-mds/nemo', '/mnt/lustre/ls1-europe-west4-a/jobs:/mnt/lustre/ls1-europe-west4-a/jobs']
  TensorBoard: ai-infra-europe-west4-tb
  TensorBoard Project: ai-infra-recipe-validation
  TensorBoard Region: europe-west4
  TensorBoard GCS: ai-infra-gcs-europe-west4


In [5]:
# =============================================================================
# STEP 2: Build torchrun Arguments
# =============================================================================

# The sbatch template exports these variables:
# - MASTER_ADDR: IP of the head node
# - MASTER_PORT: Port for distributed communication (default 29500)
# - GPUS_PER_NODE: Number of GPUs per node
# - JOB_IDENTIFIER: Unique job identifier (Slurm job ID)
# - SLURM_NNODES, SLURM_PROCID: Standard Slurm variables
# - AIP_TENSORBOARD_LOG_DIR: (VMDS sets this) GCS path for TensorBoard logs

# Use ShellSafeArg for arguments containing shell variables
# This prevents them from being quoted (which would prevent expansion)
#
# IMPORTANT: Variable expansion inside bash -c "..."
#   ${VAR}  - expands in OUTER shell (sbatch script) before srun
#   \${VAR} - expands INSIDE bash -c (after srun sets per-task vars)
#
# SLURM_PROCID is set BY srun for each task, so must use \${SLURM_PROCID}

torchrun_args = [
    xm.ShellSafeArg('--nproc-per-node=${GPUS_PER_NODE}'),
    xm.ShellSafeArg('--nnodes=${SLURM_NNODES}'),
    xm.ShellSafeArg('--node_rank=\\${SLURM_PROCID}'),  # Escaped: expands inside srun
    xm.ShellSafeArg('--rdzv_id=${JOB_IDENTIFIER}'),
    xm.ShellSafeArg('--rdzv-endpoint=${MASTER_ADDR}:${MASTER_PORT}'),
    '--rdzv-backend=static',
    f'{WORK_DIR}/{TRAINING_SCRIPT}',
]

# Add training script arguments (NeMo factory pattern)
# - explicit_log_dir: Checkpoints saved to Lustre
# - tensorboard_log_dir: TensorBoard logs saved to GCS (via AIP_TENSORBOARD_LOG_DIR)
extra_args = [
    # Checkpoints go to Lustre, TensorBoard logs go to GCS
    f"--factory='configure_recipe(explicit_log_dir={LOGS_PATH}/${{JOB_IDENTIFIER}}/, tensorboard_log_dir=${{AIP_TENSORBOARD_LOG_DIR}})'",
    "trainer.num_nodes=${SLURM_NNODES}",
    "trainer.max_steps=5",
]

# Wrap extra args in ShellSafeArg to preserve quoting
for arg in extra_args:
    torchrun_args.append(xm.ShellSafeArg(arg))

print("Torchrun arguments:")
for arg in torchrun_args:
    if isinstance(arg, xm.ShellSafeArg):
        print(f"  {arg.arg}")
    else:
        print(f"  {arg}")

Torchrun arguments:
  --nproc-per-node=${GPUS_PER_NODE}
  --nnodes=${SLURM_NNODES}
  --node_rank=\${SLURM_PROCID}
  --rdzv_id=${JOB_IDENTIFIER}
  --rdzv-endpoint=${MASTER_ADDR}:${MASTER_PORT}
  --rdzv-backend=static
  /home/abhishekbhgwt_google_com/vertexai-mds/nemo/pretrain.py
  --factory='configure_recipe(explicit_log_dir=/mnt/lustre/ls1-europe-west4-a/jobs/${JOB_IDENTIFIER}/, tensorboard_log_dir=${AIP_TENSORBOARD_LOG_DIR})'
  trainer.num_nodes=${SLURM_NNODES}
  trainer.max_steps=5


In [ ]:
# =============================================================================
# STEP 3: Create and Submit the Job
# =============================================================================

import time
timestamp = time.strftime("%Y%m%d-%H%M%S")

with xm_local.create_experiment(experiment_title=f"vtc_nemo_{timestamp}") as experiment:
    
    # Create job with torchrun as executable
    job = xm.Job(
        executable=xm.Binary(path='torchrun'),
        args=torchrun_args,
        executor=executor,
    )
    
    # Submit job
    print("Submitting job to Slurm...")
    experiment.add(xm.JobGroup(job=job))
    
    experiment_id = experiment.experiment_id
    print(f"\nExperiment ID: {experiment_id}")
    print(f"\nMonitor with:")
    print(f"  squeue --me")
    print(f"  xmanager list")
    print(f"\nLogs will be in: {WORK_DIR}/slurm-<job_id>.out")

---
## 4b. Hyperparameter Sweeps with Vertex AI TensorBoard

XManager makes it easy to run **hyperparameter sweeps** with all experiments tracked in **Vertex AI TensorBoard**.

### How it works:

1. XManager passes `--extra="tensorboard_base_output_dir=<bucket>,tensorboard_url=<tb-url>"` to sbatch
2. VMDS sets `AIP_TENSORBOARD_LOG_DIR` to a unique path per job: `gs://<bucket>/<cluster-id>/tensorboard/job-<id>/`
3. NeMo writes TensorBoard logs to `${AIP_TENSORBOARD_LOG_DIR}` (via `tensorboard_log_dir` parameter)
4. Checkpoints stay on Lustre (`explicit_log_dir`)
5. All sweep runs appear in the same Vertex AI TensorBoard dashboard

### Directory Structure

```
Lustre (checkpoints):
  /mnt/lustre/<instance>/jobs/<JOB_ID>_mbs1/  # micro_batch_size=1
  /mnt/lustre/<instance>/jobs/<JOB_ID>_mbs2/  # micro_batch_size=2

GCS (TensorBoard logs - VMDS manages):
  gs://<bucket>/<cluster-id>/tensorboard/job-<id>/  # Unique per job
```

In [6]:
# =============================================================================
# HYPERPARAMETER SWEEP EXAMPLE
# =============================================================================
# This example demonstrates how to run multiple training jobs with different
# micro batch sizes using XManager's experiment.add() pattern.

import time

# Define micro batch sizes to sweep
MICRO_BATCH_SIZES = [1, 2, 4]

print(f"Hyperparameter sweep: {len(MICRO_BATCH_SIZES)} trials")
for i, mbs in enumerate(MICRO_BATCH_SIZES):
    print(f"  Trial {i+1}: micro_batch_size={mbs}")

Hyperparameter sweep: 3 trials
  Trial 1: micro_batch_size=1
  Trial 2: micro_batch_size=2
  Trial 3: micro_batch_size=4


In [ ]:
# =============================================================================
# SUBMIT HYPERPARAMETER SWEEP WITH TENSORBOARD
# =============================================================================
# Submit one job per micro_batch_size value within a single experiment.
# - Checkpoints: LOGS_PATH/<JOB_ID>_mbs<N>/ (Lustre)
# - TensorBoard: ${AIP_TENSORBOARD_LOG_DIR} (GCS, managed by VMDS)

import asyncio

timestamp = time.strftime("%Y%m%d-%H%M%S")

async def submit_sweep():
    async with xm_local.create_experiment(experiment_title=f"vtc_nemo_sweep_{timestamp}") as experiment:
        
        for micro_batch_size in MICRO_BATCH_SIZES:
            
            # Build torchrun args with this micro_batch_size
            sweep_torchrun_args = [
                xm.ShellSafeArg('--nproc-per-node=${GPUS_PER_NODE}'),
                xm.ShellSafeArg('--nnodes=${SLURM_NNODES}'),
                xm.ShellSafeArg('--node_rank=\\${SLURM_PROCID}'),
                xm.ShellSafeArg('--rdzv_id=${JOB_IDENTIFIER}'),
                xm.ShellSafeArg('--rdzv-endpoint=${MASTER_ADDR}:${MASTER_PORT}'),
                '--rdzv-backend=static',
                f'{WORK_DIR}/{TRAINING_SCRIPT}',
                # NeMo training args:
                # - Checkpoints to Lustre (explicit_log_dir)
                # - TensorBoard logs to GCS (tensorboard_log_dir via AIP_TENSORBOARD_LOG_DIR)
                xm.ShellSafeArg(f"--factory='configure_recipe(explicit_log_dir={LOGS_PATH}/${{JOB_IDENTIFIER}}_mbs{micro_batch_size}/, tensorboard_log_dir=${{AIP_TENSORBOARD_LOG_DIR}})'"),
                xm.ShellSafeArg("trainer.num_nodes=${SLURM_NNODES}"),
                xm.ShellSafeArg("trainer.max_steps=15"),
                xm.ShellSafeArg(f"data.micro_batch_size={micro_batch_size}"),
            ]
            
            # Create job
            job = xm.Job(
                executable=xm.Binary(path='torchrun'),
                args=sweep_torchrun_args,
                executor=executor,
            )
            
            # Submit job
            print(f"Submitting job with micro_batch_size={micro_batch_size}...")
            experiment.add(xm.JobGroup(job=job))
        
        print(f"\nExperiment ID: {experiment.experiment_id}")
        print(f"Submitted {len(MICRO_BATCH_SIZES)} jobs for hyperparameter sweep")
        print(f"\nCheckpoints: {LOGS_PATH}/<JOB_ID>_mbs<N>/")
        print(f"TensorBoard: View in Vertex AI TensorBoard ({TENSORBOARD_NAME})")
        print(f"Monitor jobs: squeue --me")

# Run the async function
asyncio.get_event_loop().run_until_complete(submit_sweep())

---
## 5. Experiment Tracking

XManager tracks all experiments in a **local SQLite database**.

You can list, query, and retrieve experiment details programmatically or via CLI.

In [ ]:
# List all experiments
experiments = xm_local.list_experiments()

# Convert to DataFrame for nice display
exp_data = []
for exp in experiments[-10:]:  # Last 10 experiments
    exp_data.append({
        "ID": exp.experiment_id,
        "Title": exp._experiment_title,
    })

df = pd.DataFrame(exp_data)
print("Recent Experiments:")
display(df)

In [ ]:
# Get a specific experiment by ID
# Replace with an actual experiment ID from your list
EXPERIMENT_ID = 1768515074491

if EXPERIMENT_ID:
    exp = xm_local.get_experiment(EXPERIMENT_ID)
    print(f"Experiment: {exp._experiment_title}")
    print(f"ID: {exp.experiment_id}")
    
    work_units = exp._experiment_units
    print(f"\nWork Units: {len(work_units)}")
    for wu in work_units:
        print(f"  - Work Unit {wu.work_unit_id}")
        if hasattr(wu, '_non_local_execution_handles'):
            for handle in wu._non_local_execution_handles:
                if hasattr(handle, 'slurm_job_id'):
                    print(f"    Slurm Job ID: {handle.slurm_job_id}")
else:
    print("No experiments found. Run a job first!")

---
## 6. Job Monitoring

Monitor job status using XManager or directly via SSH commands (`squeue`, `sacct`).

In [ ]:
# Helper function to run commands on the cluster
import subprocess

def run_on_cluster(cmd):
    """Run a command on the cluster via SSH."""
    if USE_GCLOUD_SSH:
        ssh_cmd = [
            "gcloud", "compute", "ssh", LOGIN_NODE,
            "--", "-T",
            "-o", f"Hostname={SSH_HOSTNAME}",
            cmd
        ]
    else:
        ssh_cmd = ["ssh", LOGIN_NODE, cmd]
    
    result = subprocess.run(ssh_cmd, capture_output=True, text=True)
    return result.stdout, result.stderr

print("SSH helper function defined.")

In [ ]:
# Check running jobs
stdout, stderr = run_on_cluster("squeue --me")
print("Running Jobs (squeue --me):")
print(stdout if stdout.strip() else "No running jobs")

In [ ]:
# Check job status via XManager handle
if EXPERIMENT_ID:
    exp = xm_local.get_experiment(EXPERIMENT_ID)
    
    for wu in exp._experiment_units:
        print(f"Work Unit {wu.work_unit_id}:")
        
        for handle in wu._non_local_execution_handles:
            if hasattr(handle, 'slurm_job_id'):
                print(f"  Slurm Job ID: {handle.slurm_job_id}")
                
            if hasattr(handle, 'get_status'):
                try:
                    status = handle.get_status()
                    status_name = status._status.name if hasattr(status, '_status') else str(status)
                    message = status.message if hasattr(status, 'message') and status.message else ""
                    print(f"  Status: {status_name}")
                    if message:
                        print(f"  Message: {message}")
                except Exception as e:
                    print(f"  Status check failed: {e}")

---
## 7. Log Access

View job logs from the cluster. Logs are written to `work_dir/slurm-<job_id>.out`.

In [ ]:
# View logs for a specific Slurm job
SLURM_JOB_ID = "4"  # Replace with actual job ID

# Check if log file exists
stdout, _ = run_on_cluster(f"ls -la {WORK_DIR}/slurm-{SLURM_JOB_ID}.out 2>/dev/null || echo 'Log file not found'")
print(stdout)

# View last 30 lines of log
stdout, _ = run_on_cluster(f"tail -30 {WORK_DIR}/slurm-{SLURM_JOB_ID}.out 2>/dev/null || echo 'Log file not found'")
print("\nLog tail:")
print(stdout)

---
## 8. Cancel Jobs

Cancel running jobs using `scancel`.

In [ ]:
# Cancel a specific job
SLURM_JOB_ID_TO_CANCEL = "999"  # Replace with job ID to cancel

# Uncomment to actually cancel:
# stdout, stderr = run_on_cluster(f"scancel {SLURM_JOB_ID_TO_CANCEL}")
# print(f"Cancelled job {SLURM_JOB_ID_TO_CANCEL}")

print(f"To cancel job {SLURM_JOB_ID_TO_CANCEL}:")
print(f"  Uncomment the lines above, or run:")
print(f"  scancel {SLURM_JOB_ID_TO_CANCEL}")

---
## Summary

This notebook demonstrated:

| Feature | Description |
|---------|-------------|
| **Executor Setup** | Configure cluster, container, mounts, environment |
| **PyTorch/torchrun** | Distributed training with `xm.ShellSafeArg` for shell variables |
| **Cluster Configs** | Pre-configured NCCL settings per cluster type |
| **Experiment Tracking** | Track experiments in local SQLite database |
| **Job Monitoring** | Query status via `squeue`/`sacct` or XManager handles |
| **Log Access** | View logs at `work_dir/slurm-<job_id>.out` |

### CLI Usage

For command-line usage, see the launcher example:
- `examples/nemo_vtc/launcher.py` - NeMo/PyTorch training

```bash
xmanager launch examples/nemo_vtc/launcher.py -- \
    --cluster_type=hcc-a3u \
    --partition=a3u \
    --login_node=your-login-node \
    --use_gcloud_ssh \
    --ssh_hostname=your-ssh-hostname \
    --work_dir=/home/user/nemo \
    --container_image=/home/user/nemo.sqsh \
    --training_script=pretrain.py \
    --nodes=4
```